# Imports

In [2]:
import pandas as pd
import numpy as np
import os

# Opening all files in folder

In [3]:
df_dict = {}
directory = 'data_files'
for filename in os.listdir(directory):
    if filename.endswith('_decoded.csv'):
        filepath = os.path.join(directory, filename)
        df_dict[filename] = pd.read_csv(filepath)

# Keeping all important columns, renaming accordingly

In [4]:
keep_cols = {
    'Time': 'time',
    'LeftCoolantTemp': 'left thermistor',
    'RightCoolantTemp': 'right thermistor',
    'MiddleCoolantTemp': 'middle thermistor',
    'MotorTemp': 'motor temperature',
    'IGBTTemp': 'igbt temperature',
    'MotorCurrent': 'motor current',
    'MotorRPM': 'motor rpm',
    'MotorTorque': 'motor torque',
    'BamocarVDC': 'bamocar dc voltage',
    'PackCurrent': 'pack current',
    'TractiveVoltage': 'tractive voltage',
    'AccumulatorVoltage': 'pack voltage',
    'FrontRightWheelFreq': 'wheel freq'
}

cleaned_dfs = {}

for name, df in df_dict.items():
    if len(df) > 25000:
        columns_to_keep = [col for col in df.columns if col in keep_cols.keys()]
        df = df[columns_to_keep]
        df = df.rename(columns=keep_cols)
        cleaned_dfs[name] = df

# Dropping empty rows

In [8]:
ignore_cols = ['time']

for df in cleaned_dfs.values():
    data_cols = [c for c in df.columns if c not in ignore_cols]
    df = df[~((df[data_cols].fillna(0) == 0).all(axis=1))]

# Interpolating empty values

In [9]:
for key, df in cleaned_dfs.items():

    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    df = df.set_index('time')
    df = df.interpolate(method='values')
    df = df.reset_index()

    if len(df) > 0:
        timestamp_offset = df.loc[0, 'time']
        df['time'] = df['time'] - timestamp_offset

    cleaned_dfs[key] = df

# Saving processed data

In [10]:
for key, value in cleaned_dfs.items():
    value.to_csv(f'processed_data/processed_{key}', index=False)